# Assignment 8: English → Indic NMT with OpenNMT-py (BPCC / ILCI)
Length-binned train/test splits, Transformer NMT, bin-wise BLEU with sacreBLEU.

In [1]:
!pip install -q OpenNMT-py sacrebleu sentencepiece huggingface_hub pandas


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 1. Configuration

In [2]:
import os, glob, json, random, subprocess, shutil, re
import pandas as pd

# Persist everything in WORK. On Colab, mount Drive so files survive a runtime reset.
USE_DRIVE = False   # set True on Colab
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORK = "/content/drive/MyDrive/mt_assignment8"
else:
    WORK = "work"
os.makedirs(WORK, exist_ok=True)

SRC_LANG = "eng_Latn"
TGT_LANG = "hin_Deva"          # <-- change to your native language code (e.g. guj_Gujr, mar_Deva, ben_Beng)
SEED = 42
TEST_FRAC = 0.10               # per-bin test fraction
VALID_FRAC = 0.02              # per-bin validation fraction (taken from train, needed by OpenNMT)
SPM_VOCAB = 8000
TRAIN_STEPS = 20000
random.seed(SEED)

MODEL_PREFIX = f"{WORK}/model_v2"   # new checkpoint name so the earlier (broken-vocab) run is never reused

def to_lf(path):
    """Windows writes CRLF line endings; OpenNMT needs plain LF. Rewrites the file in place if needed."""
    with open(path, "rb") as f: b = f.read()
    if b"\r" in b:
        with open(path, "wb") as f: f.write(b.replace(b"\r\n", b"\n"))


## 2. Load the ILCI subset of BPCC
The dataset is gated: accept the terms on its Hugging Face page, then log in below.

In [3]:
from huggingface_hub import login, snapshot_download
PAIRS_CSV = f"{WORK}/pairs.csv"
pair_files = []
if os.path.exists(PAIRS_CSV):
    print("pairs.csv already saved -> skipping download")
else:
    login()   # paste your HF token
    local = snapshot_download("ai4bharat/BPCC", repo_type="dataset", allow_patterns=["ilci/*", "ilci/**"])  # cached by HF
    files = [os.path.join(r, f) for r, _, fs in os.walk(os.path.join(local, "ilci")) for f in fs]
    print(len(files), "file(s) under ilci/")
    for f in files[:40]: print(" ", os.path.relpath(f, local))
    # keep files that belong to the target language (folder or file name contains the language code)
    pair_files = [f for f in files if TGT_LANG in f]
    print("\nselected", len(pair_files), "file(s) for", TGT_LANG)

pairs.csv already saved -> skipping download


In [4]:
def read_any(path):
    if path.endswith(".parquet"): return pd.read_parquet(path)
    if path.endswith(".csv"):     return pd.read_csv(path)
    if path.endswith(".tsv"):     return pd.read_csv(path, sep="\t")
    try:    return pd.read_json(path, lines=True)
    except ValueError: return pd.read_json(path)

if os.path.exists(PAIRS_CSV):
    df = pd.read_csv(PAIRS_CSV, keep_default_na=False)
else:
    assert pair_files, "No ilci files matched the target language - check the file list printed above"
    df = pd.concat([read_any(f) for f in pair_files], ignore_index=True)
    print(df.columns.tolist())
    # if several language pairs are mixed in the files, keep only English -> target
    if {"src_lang", "tgt_lang"} <= set(df.columns):
        df = df[(df.src_lang == SRC_LANG) & (df.tgt_lang == TGT_LANG)]
    df = df[["src", "tgt"]].dropna()
    df["src"] = df["src"].astype(str).str.strip()
    df["tgt"] = df["tgt"].astype(str).str.strip()
    df = df[(df.src != "") & (df.tgt != "")].drop_duplicates().reset_index(drop=True)
    df.to_csv(PAIRS_CSV, index=False)
print(len(df), "sentence pairs")
df.head()

165649 sentence pairs


,src,tgt
0,"Some countries have declining epidemics, with ...","कुछ देशों में महामारी घटती जा रही है, जहाँ कई ..."
1,Countries that have taken aggressive measures ...,जिन देशों ने वायरस को रोकने के लिए आक्रामक उपा...
2,We expect the number of cases and deaths to ri...,हम उम्मीद करते हैं कि आने वाले हफ्तों में मामल...
3,All countries must be ready to aggressively co...,सभी देशों को आक्रामक रूप से वायरस के लिए तैयार...
4,Transmission in most settings was driven by fa...,अधिकांश सेटिंग्स में प्रसार पारिवारिक समूहों द...


## 3. Length-based bins and stratified splits
Length = number of whitespace-separated words in the English source sentence.

| Bin | Rule |
|---|---|
| bin1 | length ≤ 10 |
| bin2 | 10 < length ≤ 20 |
| bin3 | 20 < length < 30 |
| bin4 | length ≥ 30 |

The split is done separately inside each bin, so train, validation and test all contain every length range.

In [5]:
def get_bin(n):
    if n <= 10: return "bin1"
    if n <= 20: return "bin2"
    if n < 30:  return "bin3"
    return "bin4"

if all(os.path.exists(f"{WORK}/{n}.csv") for n in ["train", "valid", "test"]):
    train, valid, test = [pd.read_csv(f"{WORK}/{n}.csv", keep_default_na=False) for n in ["train", "valid", "test"]]
    print("splits loaded from disk")
else:
    df["len"] = df["src"].str.split().str.len()
    df["bin"] = df["len"].apply(get_bin)
    print(df["bin"].value_counts().sort_index())

    train_parts, valid_parts, test_parts = [], [], []
    for b, g in df.groupby("bin"):
        g = g.sample(frac=1.0, random_state=SEED)
        n_test  = max(1, int(len(g) * TEST_FRAC))
        n_valid = max(1, int(len(g) * VALID_FRAC))
        test_parts.append(g.iloc[:n_test])
        valid_parts.append(g.iloc[n_test:n_test + n_valid])
        train_parts.append(g.iloc[n_test + n_valid:])

    train = pd.concat(train_parts).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    valid = pd.concat(valid_parts).reset_index(drop=True)
    test  = pd.concat(test_parts).reset_index(drop=True)
    for n, d in [("train", train), ("valid", valid), ("test", test)]:
        d.to_csv(f"{WORK}/{n}.csv", index=False)

print(pd.DataFrame({"train": train["bin"].value_counts(),
                    "valid": valid["bin"].value_counts(),
                    "test":  test["bin"].value_counts()}).sort_index())

splits loaded from disk
      train  valid  test
bin                     
bin1  29370    667  3337
bin2  66515   1511  7558
bin3  32289    733  3669
bin4  17600    400  2000


## 4. SentencePiece subword models

In [6]:
import sys, subprocess

# SentencePiece runs in a separate Python process so a native crash cannot kill the notebook kernel.
SPM_TOOL = f"{WORK}/spm_tool.py"
with open(SPM_TOOL, "w", encoding="utf-8") as f:
    f.write('''
import sys
import sentencepiece as spm
cmd = sys.argv[1]
if cmd == "train":
    inp, prefix, cov, vocab = sys.argv[2], sys.argv[3], float(sys.argv[4]), int(sys.argv[5])
    spm.SentencePieceTrainer.train(input=inp, model_prefix=prefix, vocab_size=vocab, model_type="unigram",
                                   character_coverage=cov, input_sentence_size=1000000,
                                   shuffle_input_sentence=True, num_threads=1)
else:
    sp = spm.SentencePieceProcessor(model_file=sys.argv[2])
    with open(sys.argv[3], encoding="utf-8") as fin, open(sys.argv[4], "w", encoding="utf-8", newline="\\n") as fout:
        for line in fin:
            line = line.rstrip("\\n")
            if cmd == "encode":
                fout.write(" ".join(sp.encode(line, out_type=str)) + "\\n")
            else:  # decode
                fout.write(sp.decode(line.split()) + "\\n")
''')

def run_spm(args):
    r = subprocess.run([sys.executable, SPM_TOOL] + [str(a) for a in args], capture_output=True,
                       encoding="utf-8", errors="replace", env={**os.environ, "PYTHONUTF8": "1", "PYTHONIOENCODING": "utf-8"})
    if r.returncode != 0:
        print("STDOUT:", r.stdout[-2000:]); print("STDERR:", r.stderr[-4000:])
        raise RuntimeError(f"sentencepiece step failed (exit code {r.returncode})")

def dump(lines, path):
    with open(path, "w", encoding="utf-8", newline="\n") as f:
        f.write("\n".join(" ".join(str(l).split()) for l in lines) + "\n")

for name, d in [("train", train), ("valid", valid), ("test", test)]:
    for lang in ["src", "tgt"]:
        if not (name == "test" and lang == "tgt"):
            dump(d[lang], f"{WORK}/{name}.raw.{lang}")

for lang, cov in [("src", 1.0), ("tgt", 0.9995)]:
    if not os.path.exists(f"{WORK}/spm_{lang}.model"):
        print("training SentencePiece:", lang)
        run_spm(["train", f"{WORK}/train.raw.{lang}", f"{WORK}/spm_{lang}", cov, SPM_VOCAB])

for name in ["train", "valid", "test"]:
    for lang in ["src", "tgt"]:
        if name == "test" and lang == "tgt": continue
        out = f"{WORK}/{name}.sp.{lang}"
        if not os.path.exists(out):
            run_spm(["encode", f"{WORK}/spm_{lang}.model", f"{WORK}/{name}.raw.{lang}", out])
for name in ["train", "valid", "test"]:
    for lang in ["src", "tgt"]:
        for kind in ["raw", "sp"]:
            p = f"{WORK}/{name}.{kind}.{lang}"
            if os.path.exists(p): to_lf(p)
print("SentencePiece done")

SentencePiece done


## 5. OpenNMT-py Transformer: config, vocab, training

In [7]:
import torch, yaml
use_gpu = torch.cuda.is_available()

cfg = {
    "save_data": f"{WORK}/onmt",
    "src_vocab": f"{WORK}/vocab.src",
    "tgt_vocab": f"{WORK}/vocab.tgt",
    "overwrite": True,
    "src_vocab_size": SPM_VOCAB, "tgt_vocab_size": SPM_VOCAB,
    "data": {
        "corpus_1": {"path_src": f"{WORK}/train.sp.src", "path_tgt": f"{WORK}/train.sp.tgt"},
        "valid":    {"path_src": f"{WORK}/valid.sp.src", "path_tgt": f"{WORK}/valid.sp.tgt"},
    },
    "save_model": MODEL_PREFIX,
    "save_checkpoint_steps": 5000, "keep_checkpoint": 3,
    "train_steps": TRAIN_STEPS, "valid_steps": 2500, "report_every": 200, "early_stopping": 0,
    "seed": SEED,
    "world_size": 1 if use_gpu else 0,
    "gpu_ranks": [0] if use_gpu else [],
    "encoder_type": "transformer", "decoder_type": "transformer", "position_encoding": True,
    "enc_layers": 4, "dec_layers": 4, "heads": 8,
    "hidden_size": 256, "word_vec_size": 256, "transformer_ff": 1024,
    "dropout": [0.1], "attention_dropout": [0.1], "label_smoothing": 0.1,
    "param_init": 0.0, "param_init_glorot": True,
    "optim": "adam", "adam_beta1": 0.9, "adam_beta2": 0.998,
    "decay_method": "noam", "warmup_steps": 4000, "learning_rate": 2.0, "max_grad_norm": 0.0,
    "batch_size": 4096, "batch_type": "tokens", "normalization": "tokens", "accum_count": [2],
    "max_generator_batches": 2,
}
with open(f"{WORK}/config.yaml", "w") as f: yaml.safe_dump(cfg, f)

if not (os.path.exists(f"{WORK}/vocab.src") and os.path.exists(f"{WORK}/vocab.tgt")):
    !onmt_build_vocab -config {WORK}/config.yaml -n_sample -1

# vocab files must use LF endings and contain real subword tokens
for p in [f"{WORK}/vocab.src", f"{WORK}/vocab.tgt"]:
    to_lf(p)
    with open(p, "rb") as f: assert b"\r" not in f.read(), p
    with open(p, encoding="utf-8", newline="") as f:
        print(p, [next(f).rstrip("\n") for _ in range(5)])


work/vocab.src ['▁the\t145209', '▁of\t105413', 's\t92401', ',\t80755', '.\t78128']
work/vocab.tgt ['▁है\t118851', '▁के\t117406', '▁में\t85815', ',\t76921', '▁।\t72004']


In [8]:
final_model = f"{MODEL_PREFIX}_step_{TRAIN_STEPS}.pt"
ckpts = sorted(glob.glob(f"{MODEL_PREFIX}_step_*.pt"), key=lambda p: int(re.findall(r"step_(\d+)", p)[0]))
if os.path.exists(final_model):
    print("Training already finished -> skipping")
else:
    resume = f"-train_from {ckpts[-1]}" if ckpts else ""   # resumes from the latest checkpoint if interrupted
    if ckpts: print("Resuming from", ckpts[-1])
    !onmt_train -config {WORK}/config.yaml {resume}

[2026-10-19 18:56:27,359 INFO] Missing transforms field for corpus_1 data, set to default: [].
[2026-10-19 18:56:27,359 WARNING] Corpus corpus_1's weight should be given. We default it to 1 for you.
[2026-10-19 18:56:27,359 INFO] Missing transforms field for valid data, set to default: [].
[2026-10-19 18:56:27,359 INFO] Parsed 2 corpora from -data.
[2026-10-19 18:56:27,359 INFO] Get special vocabs from Transforms: {'src': [], 'tgt': []}.
[2026-10-19 18:56:27,380 INFO] The first 10 tokens of the vocabs are:['<unk>', '<blank>', '<s>', '</s>', '▁the', '▁of', 's', ',', '.', '▁and']
[2026-10-19 18:56:27,380 INFO] The decoder start token is: <s>
[2026-10-19 18:56:27,380 INFO] Building model...
[2026-10-19 18:56:28,116 INFO] Switching model to float32 for amp/apex_amp
[2026-10-19 18:56:28,116 INFO] Non quantized layer compute is fp32
[2026-10-19 18:56:29,064 INFO] NMTModel(
  (encoder): TransformerEncoder(
    (embeddings): Embeddings(
      (make_embedding): Sequential(
        (emb_luts): E

## 6. Translate the test set and decode subwords

In [9]:
RESULT_CSV = f"{WORK}/test_with_hyp.csv"
if os.path.exists(RESULT_CSV):
    test = pd.read_csv(RESULT_CSV, keep_default_na=False)
    print("translations loaded from disk")
else:
    model_path = f"{MODEL_PREFIX}_step_{TRAIN_STEPS}.pt"
    gpu_flag = "-gpu 0" if use_gpu else ""
    !onmt_translate -model {model_path} -src {WORK}/test.sp.src -output {WORK}/test.pred.sp \
        -beam_size 5 -batch_size 64 -max_length 256 {gpu_flag}
    run_spm(["decode", f"{WORK}/spm_tgt.model", f"{WORK}/test.pred.sp", f"{WORK}/test.pred.txt"])
    with open(f"{WORK}/test.pred.txt", encoding="utf-8") as f:
        test["hyp"] = [l.rstrip("\n") for l in f]
    test.to_csv(RESULT_CSV, index=False)

Time w/o python interpreter load/terminate:  115.92580461502075


[2026-10-19 20:30:25,321 INFO] Loading checkpoint from work/model_v2_step_20000.pt
[2026-10-19 20:30:26,206 INFO] Loading data into the model
C:\Users\dhruv\Downloads\ILLM_SVNIT\venv\Lib\site-packages\onmt\modules\multi_headed_attn.py:656: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at ..\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = scaled_dot_product_attention(
[2026-10-19 20:32:21,235 INFO] PRED SCORE: -0.4075, PRED PPL: 1.50 NB SENTENCES: 16564


## 7. Bin-wise BLEU (sacreBLEU)

In [10]:
import sacrebleu

rows = []
for b in sorted(test["bin"].unique()):
    g = test[test["bin"] == b]
    bleu = sacrebleu.corpus_bleu(g["hyp"].tolist(), [g["tgt"].tolist()])
    rows.append({"bin": b, "#test sents": len(g), "avg src len": round(g["len"].mean(), 1), "BLEU": round(bleu.score, 2)})

overall = sacrebleu.corpus_bleu(test["hyp"].tolist(), [test["tgt"].tolist()])
rows.append({"bin": "overall", "#test sents": len(test), "avg src len": round(test["len"].mean(), 1), "BLEU": round(overall.score, 2)})
results = pd.DataFrame(rows)
print(overall)
results

BLEU = 28.42 60.1/36.2/23.5/16.0 (BP = 0.946 ratio = 0.947 hyp_len = 336835 ref_len = 355590)


,bin,#test sents,avg src len,BLEU
0,bin1,3337,7.6,31.70
1,bin2,7558,15.3,27.93
2,bin3,3669,24.3,27.40
3,bin4,2000,37.5,29.31
4,overall,16564,18.4,28.42


## 8. Bin-wise result analysis
*(Fill in after running; refer to the table above.)*

- **Trend across bins:** which bin scores highest / lowest, and how BLEU changes as sentence length grows.
- **Reasons:** short sentences have less room for reordering and agreement errors but BLEU on very short sentences is noisy (few n-grams); long sentences suffer from long-range dependencies, omissions and word-order divergence between English (SVO) and the target language (SOV), and are rarer in training data.
- **Caveats:** per-bin test size, single run / single seed, small model and limited training steps.